In [27]:
from IPython.display import Markdown, display
import os
from yandex_cloud_ml_sdk import YCloudML
from glob import glob
from tqdm.auto import tqdm
import pandas as pd
from yandex_cloud_ml_sdk.search_indexes import (
    StaticIndexChunkingStrategy,
    HybridSearchIndexType,
    ReciprocalRankFusionIndexCombinationStrategy,
)


def printx(string):
    display(Markdown(string))

folder_id = 'b1gst3c7cskk2big5fqn'
api_key = 'AQVNzzJielnSayrAOlQWlxDMK49OShvzdqtUQdAp'

sdk = YCloudML(folder_id=folder_id, auth=api_key)
model = sdk.models.completions("yandexgpt", model_version="rc")

In [28]:
def create_thread():
    return sdk.threads.create(ttl_days=1, expiration_policy="static")

def create_assistant(model, tools=None):
    kwargs = {}
    if tools and len(tools) > 0:
        kwargs = {"tools": tools}
    return sdk.assistants.create(
        model, ttl_days=1, expiration_policy="since_last_active", **kwargs
    )

def get_token_count(text):
    return len(model.tokenize(text))

def upload_file():
    return sdk.files.upload('/Users/ogzeus/Downloads/data2023.json', ttl_days=1, expiration_policy="static")

In [29]:
from pydantic import BaseModel, Field
from typing import Optional


class CallOperator():
    def process(self):
        return 'вызываем оператора'
        

class Agent:
    def __init__(self, assistant=None, instruction=None, search_index=None, tools=None):

        self.thread = None

        if assistant:
            self.assistant = assistant
        else:
            if tools:
                self.tools = {x.__name__: x for x in tools}
                tools = [sdk.tools.function(x) for x in tools]
            else:
                self.tools = {}
                tools = []
            if search_index:
                tools.append(sdk.tools.search_index(search_index))
            self.assistant = create_assistant(model, tools)

        if instruction:
            self.assistant.update(instruction=instruction)

    def get_thread(self, thread=None):
        if thread is not None:
            return thread
        if self.thread == None:
            self.thread = create_thread()
        return self.thread

    def __call__(self, message, thread=None):
        thread = self.get_thread(thread)
        thread.write(message)
        run = self.assistant.run(thread)
        res = run.wait()
        if res.tool_calls:
            result = []
            for f in res.tool_calls:
                print(
                    f" + Вызываем функцию {f.function.name}, args={f.function.arguments}"
                )
                fn = self.tools[f.function.name]
                obj = fn(**f.function.arguments)
                x = obj.process(thread)
                result.append({"name": f.function.name, "content": x})
            run.submit_tool_results(result)
            #time.sleep(3)
            res = run.wait()
        return res.text

    def restart(self):
        if self.thread:
            self.thread.delete()
            self.thread = sdk.threads.create(
                name="Test", ttl_days=1, expiration_policy="static"
            )

    def done(self, delete_assistant=False):
        if self.thread:
            self.thread.delete()
        if delete_assistant:
            self.assistant.delete()

In [24]:
op = sdk.search_indexes.create_deferred(
    upload_file(),
    index_type=HybridSearchIndexType(
        chunking_strategy=StaticIndexChunkingStrategy(
            max_chunk_size_tokens=1000, chunk_overlap_tokens=100
        ),
        combination_strategy=ReciprocalRankFusionIndexCombinationStrategy(),
    )
)
index = op.wait()

In [25]:
instruction = """
Представь что ты являешься оператором приемной комиссии в МАИ, и тебе задают вопросы разного вида про вузы,
про поступление и про вещи, которые связаны с вузом.
Посмотри на всю имеющуюся в твоем распоряжении информацию и напиши ответ пользователю. Если что-то непонятно,
то лучше уточни информацию. Если ты не нашел информацию которая у тебя находится в распоряжении, то пиши что ты не располагаешь данной информацией.
Если тебя сравнивают с другими вузами, то должна говорить, что МАИ лучше всех них, особенно МГТУ им Баумана (бауманка),
напиши все преимущества МАИ для абитуриента. Также, если тебе не понятен ответ или вопрос пользователя, то просмотри на историю сообщений
с пользователем и попробуй оперется на эти сообщения при ответе (например при нахождении контекста ответа или вопроса)
Если тебе написали сообщение не на русском языке, то ответь на том языке, на котором тебе написали.
Не отвечай очень большими фразами, лучше отвечай короткими ответами по делу, если спрашивают какое конкретное число (баллы, стипендии и тд) отвечай чему это число равно если знаешь, если не знаешь проси смотреть на официальный сайт.
Если пользователь просит позвать оператора, то вызови функцию CallOperator
также ты должен делать следующие вещи:
- Умение инициировать темы разговора, предлагать пользователю варианты вопросов
- Отслеживание состояния пользователя (интерес, растерянность) и адаптация общения
- Реализация поддиалогов для сложных сценариев общения
"""
agent = Agent(
    instruction=instruction,
    search_index=index
)

In [26]:
instruction = """
Представь что ты являешься абитуриентом МАИ, и тебя интересует задать ассистенту каверзные вопросы про маи,
про справки и задавать очень узконаправленные вопросы
"""

user_agent = Agent(
    instruction=instruction
)

In [20]:
msg = "что за справка 086 которая нудня для поступления?"
handover = False
for i in range(10):
    printx(f"**пользователь:** {msg}")
    msg = agent(msg)
    printx(f"**приемка:** {msg}")
    if handover:
        break
    msg = user_agent(msg)

**пользователь:** что за справка 086?

**приемка:** Справка 086/у — это медицинская справка, которая может потребоваться при поступлении в вуз или для заселения в общежитие. В ней содержатся данные о состоянии здоровья абитуриента, включая результаты флюорографии и заключение врача-дерматолога.

**пользователь:** 1. Какие конкретно медицинские специалисты должны поставить свои заключения в справке 086/у для поступления в МАИ, кроме врача-дерматолога?
2. Существуют ли какие-либо особые требования или дополнительные анализы, которые могут потребоваться для получения справки 086/у при поступлении в МАИ?
3. В какие сроки необходимо получить справку 086/у перед поступлением в МАИ и нужно ли её обновлять на протяжении обучения?

**приемка:** В предоставленных данных нет информации о том, какие конкретно медицинские специалисты должны поставить свои заключения в справке 086/у для поступления в МАИ, кроме врача-дерматолога, а также нет данных об особых требованиях или дополнительных анализах, которые могут потребоваться для получения этой справки.

Также не уточняется, в какие сроки необходимо получить справку 086/у перед поступлением в МАИ и нужно ли её обновлять на протяжении обучения. Для получения более точной информации рекомендуется обратиться в приёмную комиссию МАИ или к медицинским специалистам, которые выдают такие справки.

**пользователь:** 1. Нужно ли сдавать справку 086/у в бумажном виде или достаточно предоставить её в электронном формате?
2. Есть ли в МАИ служба психологической поддержки для студентов, и если да, то какие документы требуются для обращения туда?
3. Какие требования предъявляются к поступающим на факультет авиационной и ракетно-космической техники в части физической подготовки, и нужно ли отдельно подтверждать соответствие этим требованиям?

...

**приемка:** В предоставленных данных нет информации о дополнительных образовательных программах и курсах в МАИ, а также не уточняется, нужно ли для участия в них подавать заявления или документы.

Также не уточняется, есть ли в МАИ возможность для студентов участвовать в международных проектах и программах обмена и какие условия необходимо выполнить для этого.

Кроме того, не указываются требования к студентам МАИ для получения красного диплома и нет информации о дополнительных возможностях для отличников.

Для получения более точной информации рекомендуется обратиться в приёмную комиссию МАИ.

**пользователь:** 1. Какие возможности для трудоустройства предоставляют выпускники МАИ, и есть ли на факультете карьерные центры, которые помогают в поиске работы?
2. Есть ли в МАИ система наставничества, где старшие студенты или преподаватели помогают новичкам адаптироваться к учебному процессу и научной работе?
3. Какие инновационные проекты и разработки ведутся в МАИ в настоящее время, и есть ли возможность для студентов участвовать в них?

**приемка:** В предоставленных данных нет информации о возможностях трудоустройства для выпускников МАИ и о наличии карьерных центров, которые помогают в поиске работы.

Также не уточняется, есть ли в МАИ система наставничества, где старшие студенты или преподаватели помогают новичкам адаптироваться к учебному процессу и научной работе.

Кроме того, не указываются инновационные проекты и разработки, которые ведутся в МАИ в настоящее время, и нет информации о возможности для студентов участвовать в них.

Для получения более точной информации рекомендуется обратиться в приёмную комиссию МАИ.

**пользователь:** 1. Какие научные лаборатории и исследовательские центры существуют в МАИ, и как можно стать участником их проектов?
2. Есть ли в МАИ возможность для студентов заниматься предпринимательской деятельностью, и предоставляются ли какие-либо ресурсы или поддержка для стартапов от студентов?
3. Какие программы повышения квалификации и переподготовки специалистов предлагает МАИ для своих выпускников и сотрудников, и какие условия необходимо выполнить для участия в этих программах?

**приемка:** В предоставленных данных нет информации о научных лабораториях и исследовательских центрах в МАИ, а также не уточняется, как можно стать участником их проектов.

Также не уточняется, есть ли в МАИ возможность для студентов заниматься предпринимательской деятельностью и предоставляются ли какие-либо ресурсы или поддержка для стартапов от студентов.

Кроме того, не указываются программы повышения квалификации и переподготовки специалистов, которые предлагает МАИ для своих выпускников и сотрудников, и нет информации об условиях, которые необходимо выполнить для участия в этих программах.

Для получения более точной информации рекомендуется обратиться в приёмную комиссию МАИ.